In [1]:
from Utils import NER_Utils
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from Reader import obtain_combined_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [2]:
datasets, label_list, label2id, id2label = obtain_combined_dataset(["TempEval3", "WikiWars", "MAVEN"], "BIO")

<class 'list'>
['B-DATE', 'B-DURATION', 'B-EVENT', 'B-SET', 'B-TIME', 'I-DATE', 'I-DURATION', 'I-EVENT', 'I-SET', 'I-TIME', 'O'] {'B-DATE': 0, 'B-DURATION': 1, 'B-EVENT': 2, 'B-SET': 3, 'B-TIME': 4, 'I-DATE': 5, 'I-DURATION': 6, 'I-EVENT': 7, 'I-SET': 8, 'I-TIME': 9, 'O': 10} {0: 'B-DATE', 1: 'B-DURATION', 2: 'B-EVENT', 3: 'B-SET', 4: 'B-TIME', 5: 'I-DATE', 6: 'I-DURATION', 7: 'I-EVENT', 8: 'I-SET', 9: 'I-TIME', 10: 'O'}


In [3]:
# Load tokenizer and model
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True, use_fast=True)
model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaForTokenClassification: ['lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

In [4]:
utils = NER_Utils(tokenizer, label2id, id2label)
datasets = utils.tokenize_datasets(datasets)

In [5]:
datasets

DatasetDict({
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14872
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 53539
    })
    eval: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5949
    })
})

In [6]:
training_args = TrainingArguments(
    output_dir="./results/EventTimex-NER",
    logging_dir="./logs/EventTimex-NER",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    num_train_epochs=2,
    save_total_limit=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [7]:
event_time_ner = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=utils.data_collator,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

In [8]:
hist = event_time_ner.train()

d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 53539
  Num Epochs = 2
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 6694
  Number of trainable parameters = 124063499


  0%|          | 0/6694 [00:00<?, ?it/s]

{'loss': 0.2981, 'learning_rate': 4.9253062443979686e-05, 'epoch': 0.03}
{'loss': 0.1552, 'learning_rate': 4.850612488795937e-05, 'epoch': 0.06}
{'loss': 0.1461, 'learning_rate': 4.775918733193905e-05, 'epoch': 0.09}
{'loss': 0.1388, 'learning_rate': 4.7012249775918735e-05, 'epoch': 0.12}
{'loss': 0.1394, 'learning_rate': 4.626531221989842e-05, 'epoch': 0.15}
{'loss': 0.1216, 'learning_rate': 4.55183746638781e-05, 'epoch': 0.18}
{'loss': 0.1319, 'learning_rate': 4.4771437107857785e-05, 'epoch': 0.21}
{'loss': 0.1245, 'learning_rate': 4.402449955183747e-05, 'epoch': 0.24}
{'loss': 0.123, 'learning_rate': 4.327756199581715e-05, 'epoch': 0.27}


***** Running Evaluation *****
  Num examples = 5949
  Batch size = 32


{'loss': 0.119, 'learning_rate': 4.2530624439796834e-05, 'epoch': 0.3}


...

Trainer is attempting to log a value of "              precision    recall  f1-score   support

        DATE       0.88      0.83      0.86      2619
    DURATION       0.47      0.68      0.56       455
       EVENT       0.82      0.81      0.81     14760
         SET       0.27      0.44      0.33         9
        TIME       0.52      0.54      0.53        84

   micro avg       0.81      0.81      0.81     17927
   macro avg       0.59      0.66      0.62     17927
weighted avg       0.82      0.81      0.81     17927
" of type <class 'str'> for key "eval/classification_report" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Saving model checkpoint to ./results/EventTimex-NER\checkpoint-2000
Configuration saved in ./results/EventTimex-NER\checkpoint-2000\config.json


{'eval_loss': 0.10685474425554276, 'eval_precision': 0.8124614435533621, 'eval_recall': 0.8081106710548335, 'eval_f1': 0.8102802170143745, 'eval_classification_report': '              precision    recall  f1-score   support\n\n        DATE       0.88      0.83      0.86      2619\n    DURATION       0.47      0.68      0.56       455\n       EVENT       0.82      0.81      0.81     14760\n         SET       0.27      0.44      0.33         9\n        TIME       0.52      0.54      0.53        84\n\n   micro avg       0.81      0.81      0.81     17927\n   macro avg       0.59      0.66      0.62     17927\nweighted avg       0.82      0.81      0.81     17927\n', 'eval_runtime': 36.2152, 'eval_samples_per_second': 164.268, 'eval_steps_per_second': 5.136, 'epoch': 0.6}


Model weights saved in ./results/EventTimex-NER\checkpoint-2000\pytorch_model.bin
tokenizer config file saved in ./results/EventTimex-NER\checkpoint-2000\tokenizer_config.json
Special tokens file saved in ./results/EventTimex-NER\checkpoint-2000\special_tokens_map.json


{'loss': 0.1101, 'learning_rate': 3.431431132357335e-05, 'epoch': 0.63}
{'loss': 0.1137, 'learning_rate': 3.356737376755303e-05, 'epoch': 0.66}
{'loss': 0.1149, 'learning_rate': 3.2820436211532716e-05, 'epoch': 0.69}


KeyboardInterrupt: 

In [ ]:
event_time_ner.save_model("./results/EventTimex-NER")
event_time_ner.tokenizer.save_pretrained("./results/EventTimex-NER")


In [ ]:
event_time_ner.evaluate(datasets["test"])